# Notebook 9 — Revision-01 analyses addressing reviewer points 3.3 and 3.6

Reviewer (`recenzia.md`) raised two testable points we address here:

* **(1) Section 3.3** — compare alternative functional forms $f(\omega(g))$ against the chosen $\sqrt{\omega(g)}$ on AIC/BIC.
* **(2) Section 3.6** — test whether the intercept $c$ in $M_1$ can be dropped (its 90% CI contains zero).

Same dataset, same filter ($\rho \in [0.05,1.10]$, $g \in [2,100]$ even, $N_g^{\mathrm{emp}} \ge 100$) as notebook 06. Outputs: `paper-rev-01/rev01_results.json` and a printed summary for the revised paper.

In [1]:
import math, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import integrate, stats as sstats

WORK = Path.cwd()
DATA_DIR = WORK / 'ml_data'
OUT_DIR = WORK / 'paper-rev-01'
OUT_DIR.mkdir(exist_ok=True)
C2_TWIN = 0.6601618158468695739278121100145557784326233602847334133194484233354
RHO_MIN, RHO_MAX = 0.05, 1.10

def hl_correct(g):
    if g <= 0 or g % 2: return 0.0
    n = g
    while n % 2 == 0: n //= 2
    corr, p = 1.0, 3
    while p*p <= n:
        if n % p == 0:
            corr *= (p-1)/(p-2)
            while n % p == 0: n //= p
        p += 2
    if n > 1: corr *= (n-1)/(n-2)
    return 2*C2_TWIN*corr

def omega(n):
    if n <= 1: return 0
    c, d = 0, 2
    while d*d <= n:
        if n % d == 0:
            c += 1
            while n % d == 0: n //= d
        d += 1
    if n > 1: c += 1
    return c

def sigma0(n):
    if n <= 0: return 0
    c, i = 0, 1
    while i*i <= n:
        if n % i == 0:
            c += 2 if i*i != n else 1
        i += 1
    return c

def li2(N):
    v, _ = integrate.quad(lambda t: 1.0/np.log(t)**2, 2.0, N, limit=200)
    return float(v)

In [2]:
cached = sorted(DATA_DIR.glob('gaps_N*.csv'))
assert cached, 'no cached histograms in ml_data/'
histograms = {int(f.stem.replace('gaps_N','')): pd.read_csv(f) for f in cached}

rows = []
for N, df in histograms.items():
    lnN = math.log(N); lnlnN = math.log(math.log(N)); li2_N = li2(N)
    for _, r in df.iterrows():
        g, emp = int(r['gap']), int(r['count'])
        if g < 2 or g > 100 or g % 2 or emp < 100: continue
        Cg = hl_correct(g); rho = g*Cg/lnN
        if not (RHO_MIN <= rho <= RHO_MAX): continue
        wg = omega(g)
        rows.append({
            'N': N, 'g': g,
            'log_empir': math.log(emp),
            'log_C_li': math.log(Cg*li2_N),
            'log_Cg':   math.log(Cg),
            'rho': rho,
            'omega_g':       wg,
            'sqrt_omega_g':  math.sqrt(wg),
            'log_omega_g':   math.log(wg) if wg>0 else 0.0,
            'omega_m1':      wg-1,
            'sqrt_omega_m1': math.sqrt(max(wg-1,0)),
            'sigma0_g':      sigma0(g),
            'sqrt_sigma0_g': math.sqrt(sigma0(g)),
            'lnlnN': lnlnN,
            'weight': min(1.0, emp/1000.0),
        })
D = pd.DataFrame(rows)
print(f'Dataset: n = {len(D)} points over {len(histograms)} N values, '
      f'N in [{min(histograms):.0e}, {max(histograms):.0e}]')

Dataset: n = 140 points over 25 N values, N in [1e+05, 1e+13]


In [3]:
def weighted_lstsq(X, y, w):
    ws = np.sqrt(w)
    coefs, *_ = np.linalg.lstsq(X*ws[:,None], y*ws, rcond=None)
    resid = y - X @ coefs
    rss_w = float(np.sum(w*resid**2))
    n = len(y); sigma2 = rss_w/n
    ll = -0.5*n*(math.log(2*math.pi*sigma2) + 1.0)
    return coefs, rss_w, ll

def aic_bic(ll, k, n):
    return 2*k - 2*ll, k*math.log(n) - 2*ll

def fit_model(D, feature_cols, include_intercept=True):
    y = D['log_empir'].values
    offset = D['log_C_li'].values - D['rho'].values
    target = y - offset
    cols = [D[c].values for c in feature_cols]
    if include_intercept:
        cols.append(np.ones(len(D)))
    X = np.column_stack(cols)
    coefs, rss_w, ll = weighted_lstsq(X, target, D['weight'].values)
    k = X.shape[1] + 1  # +1 for sigma^2
    aic, bic = aic_bic(ll, k, len(D))
    return dict(features=feature_cols, intercept=include_intercept,
                coefs=coefs, rss_w=rss_w, loglik=ll, k=k, aic=aic, bic=bic, n=len(D))

## (1) Alternative functional forms $f(\omega(g))$

Each candidate replaces $\sqrt{\omega(g)}$ in $M_1$; the other feature (`log log N`) and the intercept are kept. Sorted by AIC.

In [4]:
forms = [
    ('sqrt_omega_g',   'sqrt(omega)'),
    ('omega_g',        'omega'),
    ('log_omega_g',    'log(omega)'),
    ('omega_m1',       'omega - 1'),
    ('sqrt_omega_m1',  'sqrt(omega - 1)'),
    ('log_Cg',         'log C_g'),
    ('sigma0_g',       'sigma_0(g)'),
    ('sqrt_sigma0_g',  'sqrt(sigma_0(g))'),
]
results_forms = []
for col, label in forms:
    r = fit_model(D, [col, 'lnlnN'], include_intercept=True)
    r['label'] = label
    results_forms.append(r)
results_forms.sort(key=lambda r: r['aic'])
aic_min = results_forms[0]['aic']
bic_min = min(r['bic'] for r in results_forms)

print(f"{'f(omega)':>20s}  {'a':>9s} {'b':>9s} {'c':>9s}  "
      f"{'RSS_w':>8s} {'AIC':>9s} {'dAIC':>7s}  {'BIC':>9s} {'dBIC':>7s}")
for r in results_forms:
    a, b, c = r['coefs']
    print(f"{r['label']:>20s}  {a:+9.4f} {b:+9.4f} {c:+9.4f}  "
          f"{r['rss_w']:8.4f} {r['aic']:+9.3f} {r['aic']-aic_min:7.3f}  "
          f"{r['bic']:+9.3f} {r['bic']-bic_min:7.3f}")

            f(omega)          a         b         c     RSS_w       AIC    dAIC        BIC    dBIC
               omega    +0.3493   -0.2377   +0.6177    1.0978  -273.469   0.000   -261.702   0.000
          log(omega)    +0.5039   -0.2377   +0.9670    1.0978  -273.469   0.000   -261.702   0.000
           omega - 1    +0.3493   -0.2377   +0.9670    1.0978  -273.469   0.000   -261.702   0.000
     sqrt(omega - 1)    +0.3493   -0.2377   +0.9670    1.0978  -273.469   0.000   -261.702   0.000
         sqrt(omega)    +0.8433   -0.2377   +0.1237    1.0978  -273.469   0.000   -261.702   0.000
             log C_g    +0.6297   -0.1623   +0.5939    1.5401  -226.070  47.399   -214.303  47.399
    sqrt(sigma_0(g))    +0.5201   -0.2138   +0.0507    2.5778  -153.957 119.512   -142.190 119.512
          sigma_0(g)    +0.1424   -0.2162   +0.5234    2.6691  -149.081 124.387   -137.315 124.387


## (2) Two-parameter model (no intercept $c$)

Reviewer note 3.6: 90% CI for $c$ contains zero. Compare $M_1$ with and without $c$ via AIC, BIC and a likelihood-ratio test ($\chi^2_1$).

In [5]:
m1_full = fit_model(D, ['sqrt_omega_g', 'lnlnN'], include_intercept=True)
m1_noc  = fit_model(D, ['sqrt_omega_g', 'lnlnN'], include_intercept=False)

for name, r in [('M1 (a, b, c)', m1_full), ('M1_noc (a, b)', m1_noc)]:
    print(f"{name:>18s}: coefs={np.array2string(r['coefs'], precision=4, sign='+')}  "
          f"RSS_w={r['rss_w']:.4f}  AIC={r['aic']:+.3f}  BIC={r['bic']:+.3f}")

dAIC = m1_noc['aic'] - m1_full['aic']
dBIC = m1_noc['bic'] - m1_full['bic']
lr = 2*(m1_full['loglik'] - m1_noc['loglik'])
p_lr = 1 - sstats.chi2.cdf(lr, df=1)
print(f"\nDelta AIC (noc - full) = {dAIC:+.3f}")
print(f"Delta BIC (noc - full) = {dBIC:+.3f}")
print(f"Likelihood-ratio stat  = {lr:.3f} (df=1), p = {p_lr:.4f}")

      M1 (a, b, c): coefs=[+0.8433 -0.2377 +0.1237]  RSS_w=1.0978  AIC=-273.469  BIC=-261.702
     M1_noc (a, b): coefs=[+0.8552 -0.2017]  RSS_w=1.1114  AIC=-273.739  BIC=-264.915

Delta AIC (noc - full) = -0.271
Delta BIC (noc - full) = -3.213
Likelihood-ratio stat  = 1.729 (df=1), p = 0.1885


In [6]:
summary = {
    'dataset_size': len(D),
    'N_values': sorted(histograms.keys()),
    'filter': {'rho_min': RHO_MIN, 'rho_max': RHO_MAX,
               'g_range': [2, 100], 'emp_min': 100},
    'alt_forms': [
        {'label': r['label'], 'features': r['features'],
         'coefs': r['coefs'].tolist(), 'rss_w': r['rss_w'],
         'aic': r['aic'], 'bic': r['bic'],
         'daic': r['aic']-aic_min, 'dbic': r['bic']-bic_min}
        for r in results_forms
    ],
    'two_param_test': {
        'M1_full': {'coefs': m1_full['coefs'].tolist(),
                    'aic': m1_full['aic'], 'bic': m1_full['bic'],
                    'rss_w': m1_full['rss_w']},
        'M1_no_c': {'coefs': m1_noc['coefs'].tolist(),
                    'aic': m1_noc['aic'], 'bic': m1_noc['bic'],
                    'rss_w': m1_noc['rss_w']},
        'dAIC_noc_minus_full': dAIC,
        'dBIC_noc_minus_full': dBIC,
        'LR_stat': lr, 'LR_p_value': float(p_lr),
    },
}
(OUT_DIR / 'rev01_results.json').write_text(json.dumps(summary, indent=2))
print('Saved:', OUT_DIR / 'rev01_results.json')

Saved: /opt/apps/jupyter/work/paper-rev-01/rev01_results.json
